In [1]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
from tqdm.auto import tqdm

In [2]:
# ways forward: maybe the +1 is wrong; I'm reducing vars (as I don't have the slacks)
# the negated rows may always be slacks on constraint >=
# is there one row or column that I must always negate in order for this to work?
# should I try the <= standardization once more?

import importlib
importlib.reload(gu)

def run_validation(instances):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        gu.standardize_lt_to_gt(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        model.optimize()
        assert model.Status == gp.GRB.OPTIMAL
        basis = gu.read_basis(model)
        tableau, col_to_var, negated_rows = gu.read_tableau(model, basis)
        with tqdm(range(tableau.shape[1])) as bar:  # weird bug where it doesn't draw the last item...
            failures = gu.validate_corner(model, basis, tableau, col_to_var, iter(bar))
        print("   Failures:", failures)

# test the cuts on simple examples:
run_validation(list(el.get_instances().values()))

In [21]:
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2'], miplib_instances['mas76']]  # drayage-25-23
run_validation(miplib_subset)